# **CODE 8a: DATA PREPARATION (MULTI-INDEX - AUTO DOWNLOAD)**

---

**Purpose:** Split data by index category with outputs auto-downloaded to local machine

## **Input files (from Google Drive `DRIVE_FOLDER`)**

| File | Source | Description |
|------|--------|-------------|
| `dataset_all.parquet` | Code 7 | Full merged dataset (~800 MB, 347 columns, incl. `Index_Classification`) |
| `feature_reference.csv` | (project) | Drives feature selection — drops `xgb_relevant='No'` features |

## **Output files (auto-download to local Downloads folder)**

For each index category (`all`, `nifty100`, `midcap150`, `smallcap`):

| File | Contents |
|------|----------|
| `train_data_{suffix}.parquet` | 2011–2019 cleaned features + `date`, `symbol`, `conviction_label` |
| `cleaning_report_{suffix}.csv` | **Column-wise NaN/inf/blank counts before & after cleaning + method & fill value** |
| `test1_data_{suffix}.parquet` | 2020–2022 with forward-filled features + `conviction_label` |
| `test2_data_{suffix}.parquet` | 2023–2025 with forward-filled features + `conviction_label` |
| `feature_metadata_{suffix}.csv` | Feature list with categorical flag |

Plus one global file:

| File | Contents |
|------|----------|
| `data_preparation_summary.csv` | Row counts and feature counts per category |

**Expected Runtime:** 10–15 minutes + 3–5 min download

---

## **STEP 0: Mount Google Drive**

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

print("\n" + "="*80)
print("Google Drive mounted successfully!")
print("="*80)
print()
print("📂 Input files will be loaded from Google Drive")
print("📥 Output files will auto-download to your local Downloads folder")
print()

Mounted at /content/drive

Google Drive mounted successfully!

📂 Input files will be loaded from Google Drive
📥 Output files will auto-download to your local Downloads folder



## **STEP 1: Import Libraries**

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
from google.colab import files
warnings.filterwarnings('ignore')

print("="*80)
print("CODE 8a: DATA PREPARATION (MULTI-INDEX - AUTO DOWNLOAD)")
print("="*80)
print()
print("✓ Libraries imported successfully")
print(f"  Current timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()
print("📊 This version creates 4 sets of files:")
print("   1. All stocks (~540 stocks)")
print("   2. Nifty 100 (~100 stocks)")
print("   3. Nifty Midcap 150 (~150 stocks)")
print("   4. Nifty Smallcap (~250 stocks)")
print()
print("📂 Inputs: From Google Drive")
print("📥 Outputs: Auto-download to Windows Downloads folder")
print()

CODE 8a: DATA PREPARATION (MULTI-INDEX - AUTO DOWNLOAD)

✓ Libraries imported successfully
  Current timestamp: 2026-06-22 13:48:37

📊 This version creates 4 sets of files:
   1. All stocks (~540 stocks)
   2. Nifty 100 (~100 stocks)
   3. Nifty Midcap 150 (~150 stocks)
   4. Nifty Smallcap (~250 stocks)

📂 Inputs: From Google Drive
📥 Outputs: Auto-download to Windows Downloads folder



## **STEP 2: Set File Paths**

**⚠️ IMPORTANT: Update DRIVE_FOLDER to match your Google Drive structure**

In [3]:
# ============================================================================
# UPDATE THIS PATH TO YOUR GOOGLE DRIVE FOLDER
# ============================================================================

DRIVE_FOLDER = '/content/drive/MyDrive/masters/'  # ← UPDATE THIS PATH

# Input files (from Google Drive)
DATASET_FILE     = os.path.join(DRIVE_FOLDER, 'dataset_all.parquet')
FEATURE_REF_FILE = os.path.join(DRIVE_FOLDER, 'feature_reference.csv')

# Output folder (Colab local storage - temporary)
OUTPUT_FOLDER = '/content/output/'
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print("-" * 80)
print("FILE PATHS CONFIGURED")
print("-" * 80)
print()
print(f"📂 INPUT LOCATION (Google Drive):")
print(f"   {DRIVE_FOLDER}")
print()
print(f"📥 OUTPUT LOCATION (Colab - temporary):")
print(f"   {OUTPUT_FOLDER}")
print(f"   Files will be auto-downloaded after creation")
print()

# Check input files
print("Checking input files...")

for fpath, fname in [
    (DATASET_FILE,     'dataset_all.parquet'),
    (FEATURE_REF_FILE, 'feature_reference.csv'),
]:
    if os.path.exists(fpath):
        sz = os.path.getsize(fpath) / (1024 * 1024)
        print(f"✓ {fname:30s} ({sz:.2f} MB)")
    else:
        print(f"❌ ERROR: {fname} not found at {fpath}")
        print(f"   Please update DRIVE_FOLDER path above")
        raise FileNotFoundError(f"Cannot find {fpath}")

print("\n✓ All input files found")
print()

--------------------------------------------------------------------------------
FILE PATHS CONFIGURED
--------------------------------------------------------------------------------

📂 INPUT LOCATION (Google Drive):
   /content/drive/MyDrive/masters/

📥 OUTPUT LOCATION (Colab - temporary):
   /content/output/
   Files will be auto-downloaded after creation

Checking input files...
✓ dataset_all.parquet            (1495.46 MB)
✓ feature_reference.csv          (0.11 MB)

✓ All input files found



## **STEP 3: Load Dataset and Index Mapping**

In [4]:
print("-" * 80)
print("LOADING DATA FROM GOOGLE DRIVE")
print("-" * 80)

# Load main dataset (parquet — much faster than CSV for 800 MB files)
print("\nLoading dataset_all.parquet from Drive...")
df = pd.read_parquet(DATASET_FILE)
print(f"✓ Dataset loaded: {len(df):,} rows, {len(df.columns)} columns")

# Load feature reference (drives feature selection downstream)
print("\nLoading feature_reference.csv from Drive...")
feature_ref = pd.read_csv(FEATURE_REF_FILE)
print(f"✓ Feature reference loaded: {len(feature_ref):,} features catalogued")

# List of features to drop because xgb_relevant == 'No'
features_to_drop_per_ref = (
    feature_ref.loc[feature_ref['xgb_relevant'] == 'No', 'feature_name']
              .tolist())
print(f"  Features flagged xgb_relevant='No' (will be dropped): {len(features_to_drop_per_ref)}")

# Show Index_Classification counts (this column is already in dataset_all)
if 'Index_Classification' in df.columns:
    print("\nIndex classification counts (from dataset_all):")
    for idx, count in df['Index_Classification'].value_counts().items():
        print(f"  {idx}: {count:,} rows")
else:
    print("\n⚠️ Warning: Index_Classification column not found in dataset_all")

print()


--------------------------------------------------------------------------------
LOADING DATA FROM GOOGLE DRIVE
--------------------------------------------------------------------------------

Loading dataset_all.parquet from Drive...
✓ Dataset loaded: 2,417,660 rows, 345 columns

Loading feature_reference.csv from Drive...
✓ Feature reference loaded: 347 features catalogued
  Features flagged xgb_relevant='No' (will be dropped): 65

Index classification counts (from dataset_all):
  Nifty Smallcap 250: 1,059,454 rows
  Nifty Midcap 150: 639,245 rows
  Nifty 100: 450,214 rows
  Not Mapped: 268,747 rows



## **STEP 4: Data Validation**

In [5]:
print("-" * 80)
print("DATA VALIDATION")
print("-" * 80)

# Auto-detect and standardize date column
date_column = None
possible_date_cols = ['Date', 'date', 'DATE', 'trading_date', 'timestamp']

for col in possible_date_cols:
    if col in df.columns:
        date_column = col
        break

if date_column is None:
    date_candidates = [col for col in df.columns if 'date' in col.lower()]
    if date_candidates:
        date_column = date_candidates[0]
    else:
        raise ValueError("No date column found")

df['date'] = pd.to_datetime(df[date_column])
print(f"✓ Date column '{date_column}' converted to datetime")
# Drop the ORIGINAL date column if it had a different name (e.g. 'Date'),
# otherwise it survives into feature_cols and leaks into feature_metadata.
if date_column != 'date' and date_column in df.columns:
    df = df.drop(columns=[date_column])
    print(f"✓ Dropped original '{date_column}' column to avoid duplicate date feature")

# Auto-detect and standardize symbol column
symbol_column = None
possible_symbol_cols = ['symbol', 'Symbol', 'Ticker', 'ticker', 'SYMBOL', 'stock']

for col in possible_symbol_cols:
    if col in df.columns:
        symbol_column = col
        break

if symbol_column and symbol_column != 'symbol':
    df['symbol'] = df[symbol_column]
    print(f"✓ Symbol column '{symbol_column}' renamed to 'symbol'")
    # Drop the original to avoid a duplicate identifier column leaking into features
    if symbol_column in df.columns:
        df = df.drop(columns=[symbol_column])
        print(f"✓ Dropped original '{symbol_column}' column")

# Check conviction labels
if 'conviction_label' not in df.columns:
    raise ValueError("Conviction label column missing")

print(f"✓ Conviction labels found")

# Show label distribution
print("\nConviction Label Distribution (All Stocks):")
label_dist = df['conviction_label'].value_counts().sort_index()
for label, count in label_dist.items():
    pct = count / len(df) * 100
    print(f"  {label:10s}: {count:>8,} ({pct:5.2f}%)")

# Show date range
print(f"\nDate Range:")
print(f"  Earliest: {df['date'].min().strftime('%Y-%m-%d')}")
print(f"  Latest: {df['date'].max().strftime('%Y-%m-%d')}")
print(f"  Trading days: {df['date'].nunique():,}")
print(f"  Unique stocks: {df['symbol'].nunique():,}")

print()

--------------------------------------------------------------------------------
DATA VALIDATION
--------------------------------------------------------------------------------
✓ Date column 'Date' converted to datetime
✓ Dropped original 'Date' column to avoid duplicate date feature
✓ Symbol column 'Ticker' renamed to 'symbol'
✓ Dropped original 'Ticker' column
✓ Conviction labels found

Conviction Label Distribution (All Stocks):
  High      :  603,804 (24.97%)
  Ignore    : 1,521,589 (62.94%)
  Low       :  107,877 ( 4.46%)
  Medium    :  183,820 ( 7.60%)

Date Range:
  Earliest: 2007-01-02
  Latest: 2026-06-12
  Trading days: 4,799
  Unique stocks: 570



## **STEP 5: Create Index Category Filters**

In [6]:
print("-" * 80)
print("CREATING INDEX CATEGORY FILTERS")
print("-" * 80)

# Index_Classification is already a column in dataset_all (from Code 7)
# — no merge needed
if 'Index_Classification' in df.columns:
    mapped_count = df['Index_Classification'].notna().sum()
    print(f"✓ Index_Classification already present in dataset_all")
    print(f"  Rows with index classification: {mapped_count:,}")
    print(f"  Rows without classification: {len(df) - mapped_count:,}")
else:
    raise ValueError("Index_Classification column missing from dataset_all — check Code 7 output")
print()

# Define index categories based on Index_Classification column
categories = {
    'all': {
        'name': 'All Stocks',
        'filter': lambda x: pd.Series([True] * len(x)),
        'suffix': 'all'
    },
    'nifty100': {
        'name': 'Nifty 100',
        'filter': lambda x: x['Index_Classification'] == 'Nifty 100',
        'suffix': 'nifty100'
    },
    'midcap150': {
        'name': 'Nifty Midcap 150',
        'filter': lambda x: x['Index_Classification'] == 'Nifty Midcap 150',
        'suffix': 'midcap150'
    },
    'smallcap': {
        'name': 'Nifty Smallcap',
        'filter': lambda x: x['Index_Classification'].str.startswith('Nifty Smallcap', na=False) if 'Index_Classification' in x.columns else pd.Series([False] * len(x)),
        'suffix': 'smallcap'
    }
}

# Show category sizes
print("Category sizes:")
for key, config in categories.items():
    mask = config['filter'](df)
    n_rows = mask.sum()
    n_stocks = df[mask]['symbol'].nunique()
    print(f"  {config['name']:20s}: {n_rows:>10,} rows, {n_stocks:>5} unique stocks")

print()

--------------------------------------------------------------------------------
CREATING INDEX CATEGORY FILTERS
--------------------------------------------------------------------------------
✓ Index_Classification already present in dataset_all
  Rows with index classification: 2,417,660
  Rows without classification: 0

Category sizes:
  All Stocks          :  2,417,660 rows,   570 unique stocks
  Nifty 100           :    450,214 rows,   100 unique stocks
  Nifty Midcap 150    :    639,245 rows,   150 unique stocks
  Nifty Smallcap      :  1,059,454 rows,   250 unique stocks



## **STEP 6: Define Data Splitting Function**

In [7]:
print("-" * 80)
print("DEFINING DATA SPLITTING FUNCTION")
print("-" * 80)

# Define date splits (locked)
TRAIN_START = '2011-01-01'
TRAIN_END = '2019-12-31'
TEST1_START = '2020-01-01'
TEST1_END = '2022-12-31'
TEST2_START = '2023-01-01'
TEST2_END = '2025-12-31'

print(f"Date splits (locked):")
print(f"  TRAIN:  {TRAIN_START} to {TRAIN_END}")
print(f"  TEST1:  {TEST1_START} to {TEST1_END}")
print(f"  TEST2:  {TEST2_START} to {TEST2_END}")
print()

def split_and_save_category(data, category_config, output_folder):
    """
    Split data by date and save for one category.
    """
    category_name = category_config['name']
    suffix = category_config['suffix']

    print(f"\nProcessing: {category_name}")
    print(f"  {'='*76}")

    # Filter data for this category
    mask = category_config['filter'](data)
    category_data = data[mask].copy()

    print(f"  Category data: {len(category_data):,} rows, {category_data['symbol'].nunique()} stocks")

    # Split by date
    train_df = category_data[(category_data['date'] >= TRAIN_START) &
                             (category_data['date'] <= TRAIN_END)].copy()
    test1_df = category_data[(category_data['date'] >= TEST1_START) &
                             (category_data['date'] <= TEST1_END)].copy()
    test2_df = category_data[(category_data['date'] >= TEST2_START) &
                             (category_data['date'] <= TEST2_END)].copy()

    # Show split summary
    print(f"\n  Split summary:")
    print(f"    TRAIN:  {len(train_df):>8,} rows ({train_df['symbol'].nunique():>3} stocks)")
    print(f"    TEST1:  {len(test1_df):>8,} rows ({test1_df['symbol'].nunique():>3} stocks)")
    print(f"    TEST2:  {len(test2_df):>8,} rows ({test2_df['symbol'].nunique():>3} stocks)")

    # Validate splits
    if len(train_df) == 0 or len(test1_df) == 0 or len(test2_df) == 0:
        print(f"  ⚠️ WARNING: One or more splits are empty for {category_name}!")
        return None

    # Identify feature columns (exclude non-features)
    # ── Always-excluded columns (identifiers, targets, exit simulation outputs) ──
    non_feature_cols = ['date', 'symbol', 'conviction_label',
                       # TARGET LEAKAGE: this is a numeric copy of conviction_label
                       'conviction_label_numeric',
                       # IDENTIFIER LEAKAGE: Year is unseen in test (2020-2025 vs train 2011-2019)
                       'Year',
                       # Exit simulation columns (from Code 6)
                       'entry_price', 'exit_price', 'exit_date', 'days_held',
                       'annualized_return', 'peak_price', 'drawdown',
                       # Index mapping columns (merged in earlier step)
                       'Index_Classification', 'Symbol', 'Company_Name', 'Classification_Date',
                       'Mapping_Status', 'Suggested_Index', 'Market_Cap_INR_Cr', 'Rank',
                       'Exchange', 'Ticker_Used', 'Reason']

    # ── Additionally drop features marked xgb_relevant='No' in feature_reference.csv ──
    extra_drops = [c for c in features_to_drop_per_ref if c in train_df.columns]
    non_feature_cols = non_feature_cols + extra_drops
    if extra_drops:
        print(f"  Dropping {len(extra_drops)} features per feature_reference.csv "
              f"(xgb_relevant='No')")

    # Safety net: also exclude any case/whitespace variant of identifier columns
    _identifier_variants = {'date', 'symbol', 'ticker'}
    feature_cols = [col for col in train_df.columns
                    if col not in non_feature_cols
                    and col.strip().lower() not in _identifier_variants]

    # Identify categorical features
    categorical_features = []
    for col in feature_cols:
        if train_df[col].dtype == 'object' or train_df[col].dtype.name == 'category':
            categorical_features.append(col)

    print(f"\n  Features identified:")
    print(f"    Total features: {len(feature_cols)}")
    print(f"    Categorical: {len(categorical_features)}")
    print(f"    Numerical: {len(feature_cols) - len(categorical_features)}")

    # Create feature metadata
    feature_metadata = pd.DataFrame({
        'feature_name': feature_cols,
        'feature_index': range(len(feature_cols)),
        'is_categorical': [col in categorical_features for col in feature_cols]
    })

    # Prepare data for saving (features + label)
    # Carry 'date' and 'symbol' in the saved file as IDENTIFIERS (not model features).
    #   • Code 8b uses 'date' to build walk-forward folds, then drops both before training.
    #   • Code 8c uses 'date' for entry/exit timing and 'symbol' to track each stock
    #     through its ATR-based exit, then drops both before prediction.
    # The case-variant safety net in feature_cols guarantees neither is a feature.
    id_cols = [c for c in ['date', 'symbol'] if c in train_df.columns]
    save_cols = id_cols + feature_cols + ['conviction_label']

    # ════════════════════════════════════════════════════════════════════════
    # DATA CLEANING — staged, with column-wise reports at each stage
    # Strategies come from feature_reference 'nan_handling':
    #   drop_row     : if NaN/inf in this column, drop the WHOLE row
    #   forward_fill : per-stock ffill+bfill (NO inter-stock contamination)
    #   keep_nan     : leave NaN as-is (XGBoost handles NaN natively)
    # inf is always converted to NaN first (XGBoost rejects inf, accepts NaN).
    # Reports cover ONLY 2011-2025 (the three dated splits below).
    # ════════════════════════════════════════════════════════════════════════
    nan_strategy = dict(zip(feature_ref['feature_name'], feature_ref['nan_handling']))

    def resolve_strategy(col):
        raw = str(nan_strategy.get(col, 'forward_fill'))
        if 'drop_row' in raw:     return 'drop_row'
        if 'keep_nan' in raw:     return 'keep_nan'
        if 'forward_fill' in raw: return 'forward_fill'
        return 'forward_fill'   # default for anything unspecified

    col_method = {c: resolve_strategy(c) for c in feature_cols}

    ticker_col = next((c for c in ['symbol', 'Ticker', 'ticker', 'Symbol']
                       if c in train_df.columns), None)
    numeric_feats = [c for c in feature_cols if train_df[c].dtype.kind in 'fiu']

    # Per-column audit summed across the three dated splits (2011-2025 only)
    def audit_all(splits):
        rows = []
        for c in feature_cols:
            nan_ct = inf_ct = blank_ct = 0
            for sd in splits:
                sc = sd[c]
                nan_ct += int(sc.isnull().sum())
                if sc.dtype.kind in 'fiu':
                    inf_ct += int(np.isinf(sc.to_numpy(dtype='float64',
                                                       na_value=np.nan)).sum())
                else:
                    blank_ct += int((sc.astype(str).str.strip() == '').sum())
            rows.append({'feature': c, 'cleaning_method': col_method[c],
                         'n_nan': nan_ct, 'n_inf': inf_ct, 'n_blank': blank_ct})
        return pd.DataFrame(rows)

    splits = [train_df, test1_df, test2_df]
    rows_before = sum(len(s) for s in splits)

    # ── STAGE 0: PRE-HANDLING status ─────────────────────────────────────────
    print("\n  " + "="*68)
    print("  STAGE 0 — PRE-HANDLING column status (2011-2025)")
    print("  " + "="*68)
    stage0 = audit_all(splits)
    print(f"    Total rows           : {rows_before:,}")
    print(f"    Columns with NaN     : {(stage0['n_nan']>0).sum()} / {len(stage0)}")
    print(f"    Columns with inf     : {(stage0['n_inf']>0).sum()} / {len(stage0)}")
    print(f"    Total NaN cells      : {stage0['n_nan'].sum():,}")
    print(f"    Total inf cells      : {stage0['n_inf'].sum():,}")
    print("    Top 10 columns by NaN+inf:")
    s0 = stage0.copy(); s0['issues'] = s0['n_nan'] + s0['n_inf']
    print(s0.sort_values('issues', ascending=False)
            [['feature','cleaning_method','n_nan','n_inf','n_blank']].head(10)
            .to_string(index=False))

    # ── Convert inf → NaN (so drop_row catches inf too, others stay NaN) ─────
    total_inf = int(stage0['n_inf'].sum())
    for sd in splits:
        sd[numeric_feats] = sd[numeric_feats].replace([np.inf, -np.inf], np.nan)
    print(f"\n  Converted {total_inf:,} inf/-inf → NaN before handling.")

    # ── STAGE 1: DROP_ROW ────────────────────────────────────────────────────
    drop_cols = [c for c in feature_cols if col_method[c] == 'drop_row']
    print("\n  " + "="*68)
    print(f"  STAGE 1 — DROP_ROW  ({len(drop_cols)} trigger columns)")
    print("  " + "="*68)
    print(f"    Columns: {drop_cols}")
    for name, idx in [('train_df',0),('test1_df',1),('test2_df',2)]:
        sd = splits[idx]
        before = len(sd)
        present = [c for c in drop_cols if c in sd.columns]
        sd.dropna(subset=present, inplace=True)
        print(f"    {name:9s}: {before:,} → {len(sd):,} rows "
              f"(dropped {before-len(sd):,})")
    rows_after_drop = sum(len(s) for s in splits)
    stage1 = audit_all(splits)
    print(f"\n    Total rows after drop_row: {rows_after_drop:,} "
          f"(removed {rows_before - rows_after_drop:,})")
    print(f"    Remaining NaN cells      : {stage1['n_nan'].sum():,}")

    # ── STAGE 2: FORWARD_FILL (per-stock) + KEEP_NAN (left as-is) ────────────
    ff_cols = [c for c in feature_cols if col_method[c] == 'forward_fill']
    keep_cols = [c for c in feature_cols if col_method[c] == 'keep_nan']
    print("\n  " + "="*68)
    print(f"  STAGE 2 — FORWARD_FILL ({len(ff_cols)})  +  KEEP_NAN ({len(keep_cols)})")
    print("  " + "="*68)
    print(f"    forward_fill: per-stock ffill+bfill (no inter-stock data used)")
    print(f"    keep_nan    : NaN left in place (XGBoost handles natively)")
    for sd in splits:
        sd.sort_values([ticker_col, 'date'] if ticker_col else ['date'], inplace=True)
        for col in ff_cols:
            if ticker_col:
                sd[col] = sd.groupby(ticker_col)[col].transform(lambda s: s.ffill().bfill())
            else:
                sd[col] = sd[col].ffill().bfill()
        # keep_nan columns: intentionally untouched

    stage2 = audit_all(splits)
    print(f"\n    NaN after forward_fill (incl. intentional keep_nan): "
          f"{stage2['n_nan'].sum():,}")
    ff_residual = stage2[(stage2['cleaning_method']=='forward_fill') & (stage2['n_nan']>0)]
    if len(ff_residual):
        print(f"    ⚠️  {len(ff_residual)} forward_fill cols still NaN "
              f"(entire stock history was empty):")
        print(ff_residual[['feature','n_nan']].to_string(index=False))
    else:
        print(f"    ✓ All forward_fill columns fully filled")
    keep_nan_remaining = stage2[stage2['cleaning_method']=='keep_nan']['n_nan'].sum()
    print(f"    NaN intentionally kept (keep_nan cols): {keep_nan_remaining:,}")

    # ── Build STAGED column-wise report (3 stages, absolute + percentage) ────
    # Stage 0 = pre-handling | Stage 1 = after drop_row | Stage 2 = after fill
    # Percentages use that stage's row count as the denominator.
    def pct(n, denom):
        return round(n / denom * 100, 3) if denom else 0.0

    report = pd.DataFrame({'feature': feature_cols})
    report['cleaning_method'] = report['feature'].map(col_method)

    s0 = stage0.set_index('feature'); s1 = stage1.set_index('feature'); s2 = stage2.set_index('feature')

    # Stage 0 — pre-handling (denominator = rows_before)
    report['s0_nan']     = report['feature'].map(s0['n_nan'])
    report['s0_inf']     = report['feature'].map(s0['n_inf'])
    report['s0_blank']   = report['feature'].map(s0['n_blank'])
    report['s0_nan_pct'] = report['s0_nan'].apply(lambda n: pct(n, rows_before))
    report['s0_inf_pct'] = report['s0_inf'].apply(lambda n: pct(n, rows_before))

    # Stage 1 — after drop_row (denominator = rows_after_drop)
    report['s1_nan']     = report['feature'].map(s1['n_nan'])
    report['s1_inf']     = report['feature'].map(s1['n_inf'])
    report['s1_blank']   = report['feature'].map(s1['n_blank'])
    report['s1_nan_pct'] = report['s1_nan'].apply(lambda n: pct(n, rows_after_drop))
    report['s1_inf_pct'] = report['s1_inf'].apply(lambda n: pct(n, rows_after_drop))

    # Stage 2 — after forward_fill + keep_nan (denominator = rows_after_drop)
    report['s2_nan']     = report['feature'].map(s2['n_nan'])
    report['s2_inf']     = report['feature'].map(s2['n_inf'])
    report['s2_blank']   = report['feature'].map(s2['n_blank'])
    report['s2_nan_pct'] = report['s2_nan'].apply(lambda n: pct(n, rows_after_drop))
    report['s2_inf_pct'] = report['s2_inf'].apply(lambda n: pct(n, rows_after_drop))

    # Row-count context columns
    report['rows_stage0'] = rows_before
    report['rows_stage1'] = rows_after_drop
    report['rows_stage2'] = rows_after_drop

    report = report.sort_values(['cleaning_method','s0_nan'], ascending=[True, False])
    report_file = os.path.join(output_folder, f'cleaning_report_{suffix}.csv')
    report.to_csv(report_file, index=False)

    # Column-name legend printed for clarity
    print(f"\n  ✓ Staged column-wise report saved: cleaning_report_{suffix}.csv")
    print(f"    Columns: s0_* = Stage 0 (pre-handling), s1_* = Stage 1 (after drop_row),")
    print(f"             s2_* = Stage 2 (after forward_fill + keep_nan)")
    print(f"             *_pct = percentage of that stage's rows")
    print(f"    Row counts:  Stage 0 = {rows_before:,}  |  Stage 1 & 2 = {rows_after_drop:,}")
    print(f"    (covers 2011-2025 only; forward_fill is strictly per-stock)")

    # Console preview: top columns that had issues, showing all 3 stages
    preview = report[(report['s0_nan'] > 0) | (report['s0_inf'] > 0)].head(12)
    if len(preview):
        print(f"\n  Preview — columns with issues, across all 3 stages:")
        show = ['feature','cleaning_method',
                's0_nan','s0_nan_pct','s1_nan','s1_nan_pct','s2_nan','s2_nan_pct']
        print(preview[show].to_string(index=False))

    # Re-extract save DataFrames with cleaned values
    train_save = train_df[save_cols]
    test1_save = test1_df[save_cols]
    test2_save = test2_df[save_cols]

    # Define output filenames
    train_file = os.path.join(output_folder, f'train_data_{suffix}.parquet')
    test1_file = os.path.join(output_folder, f'test1_data_{suffix}.parquet')
    test2_file = os.path.join(output_folder, f'test2_data_{suffix}.parquet')
    metadata_file = os.path.join(output_folder, f'feature_metadata_{suffix}.csv')

    # Save files to Colab
    print(f"\n  Saving files to Colab...")
    train_save.to_parquet(train_file, index=False)
    print(f"    ✓ train_data_{suffix}.parquet ({len(train_save):,} rows)")

    test1_save.to_parquet(test1_file, index=False)
    print(f"    ✓ test1_data_{suffix}.parquet ({len(test1_save):,} rows)")

    test2_save.to_parquet(test2_file, index=False)
    print(f"    ✓ test2_data_{suffix}.parquet ({len(test2_save):,} rows)")

    feature_metadata.to_csv(metadata_file, index=False)
    print(f"    ✓ feature_metadata_{suffix}.csv ({len(feature_metadata)} features)")

    return {
        'category': category_name,
        'suffix': suffix,
        'train_rows': len(train_save),
        'test1_rows': len(test1_save),
        'test2_rows': len(test2_save),
        'n_features': len(feature_cols),
        'n_categorical': len(categorical_features),
        'files': [train_file, test1_file, test2_file, metadata_file, report_file]
    }

print("\n✓ Data splitting function defined")
print()

--------------------------------------------------------------------------------
DEFINING DATA SPLITTING FUNCTION
--------------------------------------------------------------------------------
Date splits (locked):
  TRAIN:  2011-01-01 to 2019-12-31
  TEST1:  2020-01-01 to 2022-12-31
  TEST2:  2023-01-01 to 2025-12-31


✓ Data splitting function defined



## **STEP 7: Process All Categories**

In [8]:
print("="*80)
print("PROCESSING ALL CATEGORIES")
print("="*80)

results = []
all_files = []

for key, config in categories.items():
    result = split_and_save_category(df, config, OUTPUT_FOLDER)
    if result:
        results.append(result)
        all_files.extend(result['files'])

print("\n" + "="*80)
print("ALL CATEGORIES PROCESSED")
print("="*80)

PROCESSING ALL CATEGORIES

Processing: All Stocks
  Category data: 2,417,660 rows, 570 stocks

  Split summary:
    TRAIN:  1,122,934 rows (570 stocks)
    TEST1:   425,780 rows (570 stocks)
    TEST2:   421,792 rows (570 stocks)
  Dropping 65 features per feature_reference.csv (xgb_relevant='No')

  Features identified:
    Total features: 275
    Categorical: 3
    Numerical: 272

  STAGE 0 — PRE-HANDLING column status (2011-2025)
    Total rows           : 1,970,506
    Columns with NaN     : 240 / 275
    Columns with inf     : 11 / 275
    Total NaN cells      : 101,949,718
    Total inf cells      : 55,825
    Top 10 columns by NaN+inf:
                          feature cleaning_method   n_nan  n_inf  n_blank
          Commodity_Alignment_60d    forward_fill 1909385      0        0
          Commodity_Alignment_20d    forward_fill 1909385      0        0
         commodity_vol_ratio_5_20    forward_fill 1909385      0        0
  commodity_momentum_accel_60_120    forward_fill 190

## **STEP 8: Create Summary Report**

In [9]:
print("\n" + "-" * 80)
print("CREATING SUMMARY REPORT")
print("-" * 80)

# Create summary dataframe
summary_data = []
for result in results:
    summary_data.append({
        'category': result['category'],
        'suffix': result['suffix'],
        'train_rows': result['train_rows'],
        'test1_rows': result['test1_rows'],
        'test2_rows': result['test2_rows'],
        'total_rows': result['train_rows'] + result['test1_rows'] + result['test2_rows'],
        'n_features': result['n_features'],
        'n_categorical': result['n_categorical']
    })

summary_df = pd.DataFrame(summary_data)

# Save summary
summary_file = os.path.join(OUTPUT_FOLDER, 'data_preparation_summary.csv')
summary_df.to_csv(summary_file, index=False)
all_files.append(summary_file)

print(f"\n✓ Summary created")
print(f"\nSummary:")
display(summary_df)

print(f"\nTotal files created: {len(all_files)}")
print(f"Files location: {OUTPUT_FOLDER}")
print()


--------------------------------------------------------------------------------
CREATING SUMMARY REPORT
--------------------------------------------------------------------------------

✓ Summary created

Summary:


,category,suffix,train_rows,test1_rows,test2_rows,total_rows,n_features,n_categorical
0,All Stocks,all,1086662,425780,421792,1934234,275,3
1,Nifty 100,nifty100,204764,74700,74000,353464,275,3
2,Nifty Midcap 150,midcap150,286739,112048,110999,509786,275,3
3,Nifty Smallcap,smallcap,479555,186746,184999,851300,275,3



Total files created: 21
Files location: /content/output/



## **STEP 9: Auto-Download All Files**

**📥 All output files will now download to your Windows Downloads folder**

In [10]:
print("="*80)
print("AUTO-DOWNLOADING OUTPUT FILES")
print("="*80)
print()
print(f"📥 Downloading {len(all_files)} files to your Windows Downloads folder...")
print()
print("⚠️ Do NOT close browser during download!")
print()
print("Files being downloaded:")
print()

# Download each file
for i, filepath in enumerate(all_files, 1):
    filename = os.path.basename(filepath)
    file_size_mb = os.path.getsize(filepath) / (1024 * 1024)

    print(f"{i:2d}. {filename:40s} ({file_size_mb:>8.2f} MB)... ", end="")

    try:
        files.download(filepath)
        print("✓")
    except Exception as e:
        print(f"✗ ERROR: {e}")

print()
print("="*80)
print("✅ ALL FILES DOWNLOADED!")
print("="*80)
print()
print(f"📁 Check your Windows Downloads folder for {len(all_files)} CSV files")
print(f"   Default location: C:\\Users\\YourName\\Downloads\\")
print()

AUTO-DOWNLOADING OUTPUT FILES

📥 Downloading 21 files to your Windows Downloads folder...

⚠️ Do NOT close browser during download!

Files being downloaded:

 1. train_data_all.parquet                   (  680.96 MB)... 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓
 2. test1_data_all.parquet                   (  280.68 MB)... 

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✓
21. data_preparation_summary.csv             (    0.00 MB)... 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✓

✅ ALL FILES DOWNLOADED!

📁 Check your Windows Downloads folder for 21 CSV files
   Default location: C:\Users\YourName\Downloads\



## **STEP 10: Completion Summary**

In [11]:
print("="*80)
print("CODE 8a COMPLETE!")
print("="*80)
print()
print("📊 Files downloaded for each category:")
print()

for result in results:
    print(f"  {result['category']}:")
    print(f"    train_data_{result['suffix']}.csv        ({result['train_rows']:>8,} rows)")
    print(f"    test1_data_{result['suffix']}.csv        ({result['test1_rows']:>8,} rows)")
    print(f"    test2_data_{result['suffix']}.csv        ({result['test2_rows']:>8,} rows)")
    print(f"    feature_metadata_{result['suffix']}.csv  ({result['n_features']:>3} features)")
    print()

print(f"  Summary:")
print(f"    data_preparation_summary.csv")
print()

print(f"Total: {len(all_files)} files in your Downloads folder")
print()

print("="*80)
print("NEXT STEPS")
print("="*80)
print()
print("1. 📂 ORGANIZE YOUR FILES:")
print("   Create folders in Downloads and organize by category")
print()
print("2. 💾 CREATE BACKUP:")
print("   Copy all files to external drive or cloud storage")
print()
print("3. 🚀 RUN CODE 8b:")
print("   Upload train_data_all.csv and feature_metadata_all.csv")
print("   to new Colab notebook for model training")
print()
print("💡 TIP: Start with 'All Stocks' to validate pipeline,")
print("   then train specialized models for each index category")
print()

CODE 8a COMPLETE!

📊 Files downloaded for each category:

  All Stocks:
    train_data_all.csv        (1,086,662 rows)
    test1_data_all.csv        ( 425,780 rows)
    test2_data_all.csv        ( 421,792 rows)
    feature_metadata_all.csv  (275 features)

  Nifty 100:
    train_data_nifty100.csv        ( 204,764 rows)
    test1_data_nifty100.csv        (  74,700 rows)
    test2_data_nifty100.csv        (  74,000 rows)
    feature_metadata_nifty100.csv  (275 features)

  Nifty Midcap 150:
    train_data_midcap150.csv        ( 286,739 rows)
    test1_data_midcap150.csv        ( 112,048 rows)
    test2_data_midcap150.csv        ( 110,999 rows)
    feature_metadata_midcap150.csv  (275 features)

  Nifty Smallcap:
    train_data_smallcap.csv        ( 479,555 rows)
    test1_data_smallcap.csv        ( 186,746 rows)
    test2_data_smallcap.csv        ( 184,999 rows)
    feature_metadata_smallcap.csv  (275 features)

  Summary:
    data_preparation_summary.csv

Total: 21 files in your Downloa

## **STEP 11: Per-Ticker Cleaning Diagnostic** *(re-runnable)*

Inspect how the 3-stage cleaning treats a **single ticker**, stage by stage.

**How to use:**
1. Run the setup cell once (defines the diagnostic function — reuses `df` and `feature_ref` already in memory; does NOT re-run the pipeline).
2. Change `TICKER` in the run cell and re-run *just that cell* as many times as you like.
3. For each ticker it prints stage-by-stage NaN/inf and saves 4 CSVs:
   `diag_{ticker}_stage0_raw.csv`, `_stage1_after_droprow.csv`,
   `_stage2_after_fill.csv`, and `_summary.csv` — all auto-downloaded.

In [12]:
# ============================================================================
# DIAGNOSTIC SETUP — run ONCE. Reuses global `df` and `feature_ref`.
# Replays the exact 3-stage cleaning for a single ticker.
# ============================================================================
from google.colab import files as _dl_files

# Date splits (same as pipeline)
_TRAIN_START, _TRAIN_END = '2011-01-01', '2019-12-31'
_TEST1_START, _TEST1_END = '2020-01-01', '2022-12-31'
_TEST2_START, _TEST2_END = '2023-01-01', '2025-12-31'

# Same non-feature / identifier exclusions as the pipeline
# Must match the pipeline's non_feature_cols exactly so the feature list matches
_NON_FEATURE = {
    'date', 'symbol', 'conviction_label', 'conviction_label_numeric', 'Year',
    'entry_price', 'exit_price', 'exit_date', 'days_held',
    'annualized_return', 'peak_price', 'drawdown',
    'Index_Classification', 'Symbol', 'Company_Name', 'Classification_Date',
    'Mapping_Status', 'Suggested_Index', 'Market_Cap_INR_Cr', 'Rank',
    'Exchange', 'Ticker_Used', 'Reason'
}
_ID_VARIANTS = {'date','symbol','ticker'}

def _diag_feature_cols():
    cols = [c for c in df.columns
            if c not in _NON_FEATURE
            and c.strip().lower() not in _ID_VARIANTS]
    # drop xgb_relevant == 'No'
    xgb_no = set(feature_ref.loc[feature_ref['xgb_relevant'] == 'No', 'feature_name'])
    cols = [c for c in cols if c not in xgb_no]
    return cols

_diag_nan_strategy = dict(zip(feature_ref['feature_name'], feature_ref['nan_handling']))
def _diag_strategy(col):
    raw = str(_diag_nan_strategy.get(col, 'forward_fill'))
    if 'drop_row' in raw:     return 'drop_row'
    if 'keep_nan' in raw:     return 'keep_nan'
    if 'forward_fill' in raw: return 'forward_fill'
    return 'forward_fill'

def _col_status(frame, cols):
    """Per-column nan/inf/blank for a single-ticker frame."""
    rows = []
    n = len(frame)
    for c in cols:
        if c not in frame.columns:
            continue
        s = frame[c]
        nan = int(s.isnull().sum())
        if s.dtype.kind in 'fiu':
            inf = int(np.isinf(s.to_numpy(dtype='float64', na_value=np.nan)).sum())
            blank = 0
        else:
            inf = 0
            blank = int((s.astype(str).str.strip() == '').sum())
        rows.append({'feature': c, 'cleaning_method': _diag_strategy(c),
                     'n_rows': n, 'n_nan': nan, 'n_inf': inf, 'n_blank': blank,
                     'nan_pct': round(nan / n * 100, 3) if n else 0.0,
                     'inf_pct': round(inf / n * 100, 3) if n else 0.0})
    return pd.DataFrame(rows)

def diagnose_ticker(ticker, download=True, preview_rows=8):
    """Replay the 3 cleaning stages for ONE ticker. Saves + downloads 4 CSVs."""
    print("="*78)
    print(f"CLEANING DIAGNOSTIC FOR TICKER: {ticker}")
    print("="*78)

    if 'symbol' not in df.columns:
        print("⚠️  'symbol' column not found in df."); return
    tdf = df[df['symbol'] == ticker].copy()
    if len(tdf) == 0:
        avail = df['symbol'].unique()[:10]
        print(f"⚠️  Ticker '{ticker}' not found. Examples: {list(avail)}")
        return

    # Restrict to 2011-2025
    tdf['date'] = pd.to_datetime(tdf['date'])
    tdf = tdf[(tdf['date'] >= _TRAIN_START) & (tdf['date'] <= _TEST2_END)]
    tdf = tdf.sort_values('date')
    fcols = _diag_feature_cols()
    print(f"Rows for {ticker} (2011-2025): {len(tdf):,}   |   features: {len(fcols)}")
    print(f"Date range: {tdf['date'].min().date()} → {tdf['date'].max().date()}")

    numeric_feats = [c for c in fcols if tdf[c].dtype.kind in 'fiu']

    # ── STAGE 0: raw ──
    s0 = _col_status(tdf, fcols)
    stage0_frame = tdf.copy()

    # convert inf → NaN
    tdf[numeric_feats] = tdf[numeric_feats].replace([np.inf, -np.inf], np.nan)

    # ── STAGE 1: drop_row ──
    drop_cols = [c for c in fcols if _diag_strategy(c) == 'drop_row' and c in tdf.columns]
    before = len(tdf)
    tdf = tdf.dropna(subset=drop_cols)
    s1 = _col_status(tdf, fcols)
    stage1_frame = tdf.copy()
    print(f"\nStage 1 (drop_row on {len(drop_cols)} cols): {before:,} → {len(tdf):,} rows")

    # ── STAGE 2: forward_fill (per this ticker) + keep_nan ──
    ff_cols = [c for c in fcols if _diag_strategy(c) == 'forward_fill']
    for c in ff_cols:
        tdf[c] = tdf[c].ffill().bfill()   # single ticker → no inter-stock issue
    s2 = _col_status(tdf, fcols)
    stage2_frame = tdf.copy()

    # ── Summary: per-column across all 3 stages ──
    summary = s0[['feature','cleaning_method']].copy()
    summary['s0_nan'] = s0['n_nan'].values; summary['s0_nan_pct'] = s0['nan_pct'].values
    summary['s0_inf'] = s0['n_inf'].values
    s1i = s1.set_index('feature'); s2i = s2.set_index('feature')
    summary['s1_nan'] = summary['feature'].map(s1i['n_nan'])
    summary['s1_nan_pct'] = summary['feature'].map(s1i['nan_pct'])
    summary['s2_nan'] = summary['feature'].map(s2i['n_nan'])
    summary['s2_nan_pct'] = summary['feature'].map(s2i['nan_pct'])
    summary['rows_s0'] = len(stage0_frame)
    summary['rows_s1'] = len(stage1_frame)
    summary['rows_s2'] = len(stage2_frame)
    summary = summary.sort_values(['cleaning_method','s0_nan'], ascending=[True, False])

    print("\nPer-column NaN across stages (cols that had issues):")
    issues = summary[(summary['s0_nan'] > 0) | (summary['s0_inf'] > 0)]
    show = ['feature','cleaning_method','s0_nan','s0_nan_pct',
            's1_nan','s1_nan_pct','s2_nan','s2_nan_pct']
    print(issues[show].to_string(index=False) if len(issues)
          else "  (no NaN/inf in any feature for this ticker)")

    # ── Save 4 CSVs ──
    safe = ticker.replace('.','_').replace('/','_')
    out = {
        f'diag_{safe}_stage0_raw.csv'           : stage0_frame[['date'] + fcols],
        f'diag_{safe}_stage1_after_droprow.csv' : stage1_frame[['date'] + fcols],
        f'diag_{safe}_stage2_after_fill.csv'    : stage2_frame[['date'] + fcols],
        f'diag_{safe}_summary.csv'              : summary,
    }
    print("\nSaving CSVs:")
    for fname, frame in out.items():
        fpath = os.path.join(OUTPUT_FOLDER, fname)
        frame.to_csv(fpath, index=False)
        print(f"  ✓ {fname} ({len(frame):,} rows)")
        if download:
            try: _dl_files.download(fpath)
            except Exception as e: print(f"    (download skipped: {e})")

    print("\n✓ Diagnostic complete. Change TICKER below and re-run the next cell.")
    return summary

print("✓ Diagnostic ready. Use diagnose_ticker('TICKER') in the next cell.")

✓ Diagnostic ready. Use diagnose_ticker('TICKER') in the next cell.


In [13]:
# ============================================================================
# CHANGE TICKER AND RE-RUN THIS CELL  (no need to re-run the pipeline)
# ============================================================================
TICKER = 'ACC.NS'      # ← change this and re-run just this cell

_ = diagnose_ticker(TICKER, download=True)

CLEANING DIAGNOSTIC FOR TICKER: ACC.NS
⚠️  Ticker 'ACC.NS' not found. Examples: ['3IINFOLTD', 'RAMCOCEM', 'RALLIS', 'RADICO', 'PVRINOX', 'ASHOKLEY', 'PTC', 'PRSMJOHNSN', 'RAYMOND', 'PRICOLLTD']


## **STEP 12: Commodity NaN Leak Verification** *(diagnostic)*

Pinpoints why `commodity_return` NaN% drops after forward-fill. Run after the pipeline.
Checks, on the actual `df`: (a) how many tickers are entirely NaN vs partial for
`commodity_return_1d`, and (b) whether a correct per-ticker ffill leaves all-NaN tickers untouched.

In [14]:
# ============================================================================
# Verify commodity_return NaN behaviour on the ACTUAL df (2011-2025)
# ============================================================================
_col = 'commodity_return_1d'
if _col not in df.columns:
    print(f"{_col} not in df")
else:
    _d = df[(pd.to_datetime(df['date']) >= '2011-01-01') &
            (pd.to_datetime(df['date']) <= '2025-12-31')].copy()

    # Per-ticker NaN fraction for the commodity column
    per_ticker = _d.groupby('symbol')[_col].apply(lambda s: s.isnull().mean())
    n_all_nan   = int((per_ticker == 1.0).sum())     # entirely NaN
    n_no_nan    = int((per_ticker == 0.0).sum())     # never NaN
    n_partial   = int(((per_ticker > 0) & (per_ticker < 1)).sum())
    print(f"Tickers total          : {per_ticker.shape[0]}")
    print(f"  entirely NaN (unmapped): {n_all_nan}")
    print(f"  never  NaN             : {n_no_nan}")
    print(f"  partial NaN            : {n_partial}")
    print()

    total = len(_d)
    nan_before = int(_d[_col].isnull().sum())
    print(f"Rows total   : {total:,}")
    print(f"NaN before   : {nan_before:,} ({nan_before/total*100:.2f}%)")

    # Correct per-ticker ffill+bfill
    _d = _d.sort_values(['symbol','date'])
    filled = _d.groupby('symbol')[_col].transform(lambda s: s.ffill().bfill())
    nan_after = int(filled.isnull().sum())
    print(f"NaN after per-ticker ffill+bfill: {nan_after:,} ({nan_after/total*100:.2f}%)")
    print()

    # Rows belonging to entirely-NaN tickers — these MUST remain NaN
    all_nan_tickers = per_ticker[per_ticker == 1.0].index
    rows_all_nan = int(_d['symbol'].isin(all_nan_tickers).sum())
    still_nan_in_those = int(filled[_d['symbol'].isin(all_nan_tickers).values].isnull().sum())
    print(f"Rows in entirely-NaN tickers     : {rows_all_nan:,}")
    print(f"  still NaN after ffill (should = above): {still_nan_in_those:,}")
    if rows_all_nan == still_nan_in_those:
        print("  ✓ CORRECT — unmapped tickers stayed NaN (no inter-stock leak)")
    else:
        print("  ✗ LEAK — unmapped tickers got filled from other tickers!")
    print()
    print("INTERPRETATION:")
    print(f"  If 'NaN after' ≈ rows-in-all-NaN-tickers %, the fill is correct and the")
    print(f"  remaining NaN are genuinely unmapped stocks.")
    print(f"  If 'NaN after' is near 0% but many tickers are entirely NaN, there is a leak.")

Tickers total          : 570
  entirely NaN (unmapped): 464
  never  NaN             : 0
  partial NaN            : 106

Rows total   : 1,970,506
NaN before   : 1,909,385 (96.90%)
NaN after per-ticker ffill+bfill: 1,579,922 (80.18%)

Rows in entirely-NaN tickers     : 1,579,922
  still NaN after ffill (should = above): 1,579,922
  ✓ CORRECT — unmapped tickers stayed NaN (no inter-stock leak)

INTERPRETATION:
  If 'NaN after' ≈ rows-in-all-NaN-tickers %, the fill is correct and the
  remaining NaN are genuinely unmapped stocks.
  If 'NaN after' is near 0% but many tickers are entirely NaN, there is a leak.


## **STEP 13: Conviction Label NaN Audit Per Output File**

Counts how many rows in each saved train/test parquet have a NaN `conviction_label`.
NaN labels are expected for recent dates (no completed exit yet) and for early dates.
These rows must be dropped before training in Code 8b (it does so as a safety net).

In [15]:
# ============================================================================
# Count NaN conviction_label in each output parquet (per category)
# ============================================================================
import glob

print("="*78)
print("CONVICTION LABEL NaN AUDIT — per output file")
print("="*78)

parquet_files = sorted(glob.glob(os.path.join(OUTPUT_FOLDER, '*_data_*.parquet')))
if not parquet_files:
    print("No train/test parquet files found in output folder.")
else:
    rows = []
    for fpath in parquet_files:
        fname = os.path.basename(fpath)
        dfx = pd.read_parquet(fpath, columns=['conviction_label'])
        total = len(dfx)
        n_nan = int(dfx['conviction_label'].isnull().sum())
        rows.append({
            'file'        : fname,
            'total_rows'  : total,
            'nan_labels'  : n_nan,
            'nan_pct'     : round(n_nan / total * 100, 2) if total else 0.0,
            'valid_labels': total - n_nan,
        })
    audit = pd.DataFrame(rows)
    print(audit.to_string(index=False))
    print()
    print("Note: rows with NaN conviction_label cannot be used for training/scoring.")
    print("      Code 8b drops them automatically before fitting the model.")

    audit_path = os.path.join(OUTPUT_FOLDER, 'label_nan_audit.csv')
    audit.to_csv(audit_path, index=False)
    try:
        from google.colab import files as _f
        _f.download(audit_path)
        print(f"\n✓ Saved & downloaded: label_nan_audit.csv")
    except Exception:
        print(f"\n✓ Saved: label_nan_audit.csv")

CONVICTION LABEL NaN AUDIT — per output file
                        file  total_rows  nan_labels  nan_pct  valid_labels
      test1_data_all.parquet      425780           0      0.0        425780
test1_data_midcap150.parquet      112048           0      0.0        112048
 test1_data_nifty100.parquet       74700           0      0.0         74700
 test1_data_smallcap.parquet      186746           0      0.0        186746
      test2_data_all.parquet      421792           0      0.0        421792
test2_data_midcap150.parquet      110999           0      0.0        110999
 test2_data_nifty100.parquet       74000           0      0.0         74000
 test2_data_smallcap.parquet      184999           0      0.0        184999
      train_data_all.parquet     1086662           0      0.0       1086662
train_data_midcap150.parquet      286739           0      0.0        286739
 train_data_nifty100.parquet      204764           0      0.0        204764
 train_data_smallcap.parquet      479555   

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ Saved & downloaded: label_nan_audit.csv
